In [0]:
#Notes
#Check the presence of outliers
#Check multi-collinearity
#Explain method used to replace missing valeus

In [0]:
pip install missingno

In [0]:
import pyspark
import pandas as pd
import seaborn as sns
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.functions import col,when,lit,sum,round, expr,mean, stddev
import matplotlib.pyplot as plt
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler,OneHotEncoderModel, MinMaxScaler
from pyspark.ml import Pipeline
from pyspark.sql import SparkSession
from pyspark.ml.linalg import Vectors
from pyspark.ml.classification import RandomForestClassifier,LogisticRegression,MultilayerPerceptronClassifier
from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType,DoubleType
import numpy as np
from pyspark.sql import SQLContext
from pyspark.sql.functions import col, regexp_replace,split
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import missingno as msno
from sklearn.impute import KNNImputer

#EDN

In [0]:
data = pd.read_csv('bank_marketing_data.csv',sep=';')
data.drop('Unnamed: 0', axis=1, inplace=True)
print('There are {} rows and {} columns'.format(data.shape[0], data.shape[1]))
print()
dataColumns = data.columns
print(dataColumns)

data2 = spark.createDataFrame(data)#.drop('Unnamed: 0')

#Find unique counts for each column
df_unique = spark.createDataFrame([Row(column=c, unique_count=data2.select(c).dropDuplicates().count()) for c in dataColumns])
print()
print('The below lists all the columns and their unique counts')
df_unique.orderBy("unique_count", ascending=False).display()

#Split into categorical and contionous columns using the unique counts
contionousColumns =  df_unique.filter(col('unique_count')>32)
categoricalColumns =  df_unique.filter(col('unique_count')<=32)

print()
print('The below lists all the contionous columns')
print()
contionousColumns.display()
print('The below lists all the categorical columns')
print()
categoricalColumns.display()

In [0]:
data.isnull().sum()

In [0]:
# Visualize missing values as a heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(data.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

In [0]:
# Bar plot of missing values for each column
missing_values = data.isnull().sum()
missing_values = missing_values[missing_values > 0]
missing_values.sort_values(inplace=True)

plt.figure(figsize=(14, 10))
missing_values.plot(kind='barh', color='orange')
plt.title('Missing Values Count by Column', fontsize=12)
plt.xlabel('Number of Missing Values', fontsize=10)
plt.ylabel('Columns', fontsize=10)
plt.xticks(fontsize=8)
plt.yticks(fontsize=8)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [0]:
missing_pct = (data.isnull().sum() / len(data) * 100).round(2)
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=True)

plt.figure(figsize=(10, max(6, len(missing_pct) * 0.4)))  # auto-adjust height
bars = plt.barh(missing_pct.index, missing_pct, color='coral', height=0.7)

plt.title('Percentage of Missing Values by Column', fontsize=14, pad=15)
plt.xlabel('Missing (%)', fontsize=11)

# Add percentage labels
for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.4, bar.get_y() + bar.get_height()/2,
             f'{width}%', va='center', fontsize=9)

plt.gca().invert_yaxis()
plt.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming:
# - data is your pandas DataFrame
# - continuousColumns is a Spark DataFrame with one column named 'column'
#   containing the names of numeric features

continuousCol = contionousColumns.select('column').rdd.flatMap(lambda x: x).collect()

# Optional: filter only columns that actually exist in the DataFrame
continuousCol = [col for col in continuousCol if col in data.columns]

for col in continuousCol:
    # Create figure with two subplots vertically
    fig, (ax1, ax2) = plt.subplots(2, 1, 
                                  figsize=(15, 7), 
                                  height_ratios=[3, 1],
                                  gridspec_kw={'hspace': 0.35})
    
    # Histogram + KDE
    sns.histplot(data=data, x=col, bins=30, kde=True, ax=ax1, stat='density')
    ax1.set_title(f'Distribution of {col}', fontsize=14, fontweight='bold')
    #ax1.set_xlabel(col, fontsize=12)
    ax1.set_ylabel('Density', fontsize=12)
    
    # Boxplot underneath (horizontal)
    sns.boxplot(x=data[col], ax=ax2, width=0.4, fliersize=4)
    ax2.set_xlabel(col, fontsize=12)
    #ax2.set_title(f'Boxplot of {col}', fontsize=11)
    plt.show()
    print()

In [0]:
categoricalCol = categoricalColumns.select('column').rdd.flatMap(lambda x: x).collect()
for col in categoricalCol:
    if col in [ 'job','day']:
        plt.figure(figsize=(20,5))
        sns.countplot(x=col, data=data)
        plt.title('{} Variable Distribution'.format(col))
        plt.xlabel(col)
        plt.ylabel('Count')
        plt.show()
        print()
    else:
        plt.figure(figsize=(8,5))
        sns.countplot(x=col, data=data)
        plt.title('{} Variable Distribution'.format(col))
        plt.xlabel(col)
        plt.ylabel('Count')
        plt.show()
        print()



#Data Engineering

In [0]:
data['target'] = data['target'].replace({'yes': 1, 'no': 0})

In [0]:
data = pd.get_dummies(
    data,
    drop_first=True,          
    columns=[i for i in categoricalCol if 'target' not in i],  # only convert these columns
    dtype=int                
)

In [0]:
#impute missing values
imputer = KNNImputer(
    n_neighbors=2,
    weights="uniform",     
    add_indicator=False    
)

numeric_cols = [i for i in data.columns]

df_numeric_imputed = pd.DataFrame(
    imputer.fit_transform(data[numeric_cols]),
    columns=numeric_cols,
    index=data.index
)

In [0]:
#Normalize your dataset
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
data_Normalized = pd.DataFrame(scaler.fit_transform(df_numeric_imputed), columns=df_numeric_imputed.columns)

In [0]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import VectorAssembler

# Convert pandas DataFrame to Spark DataFrame
data_spark = spark.createDataFrame(data_Normalized)

# Prepare features: all columns except 'target'
feature_cols = [col for col in data_spark.columns if col != 'target']

# Assemble features into a vector column
assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')
data_assembled = assembler.transform(data_spark)

# Rename 'target' to 'label' for PySpark ML
data_ml = data_assembled.withColumnRenamed('target', 'label')
train_data, test_data = data_ml.randomSplit([0.80, 0.2], seed=1234)



In [0]:
def specifySubsetOfFeaturesToUse(data_Normalized,newFeatures):
    from pyspark.ml.classification import LogisticRegression
    from pyspark.ml.feature import VectorAssembler

    # Convert pandas DataFrame to Spark DataFrame
    data_spark = spark.createDataFrame(data_Normalized)

    # Prepare features: all columns except 'target'
    feature_cols = [col for col in newFeatures]

    # Assemble features into a vector column
    assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')
    data_assembled = assembler.transform(data_spark)

    # Rename 'target' to 'label' for PySpark ML
    data_ml = data_assembled.withColumnRenamed('target', 'label')
    train_data, test_data = data_ml.randomSplit([0.80, 0.2], seed=1234)
    return train_data, test_data


In [0]:
#ClassificationReport
def classificationReport(model,test_data):
    # Get predictions on the training data
    predictions = model.transform(test_data)
    from sklearn.metrics import classification_report, confusion_matrix

    y_true = predictions.select(['label']).collect()
    y_pred = predictions.select(['prediction']).collect()

    print(classification_report(y_true, y_pred))

    print(confusion_matrix(y_true, y_pred))


In [0]:
#balance Dataset
def balanceDataset(df):
    from sklearn.utils import resample
    df = df.toPandas()
    # Separate majority and minority classes
    df_majority = df[df.label==0]
    df_minority = df[df.label==1]

    # Upsample minority class
    df_minority_upsampled = resample(df_minority, 
                                 replace=True,     # sample with replacement
                                 n_samples=len(df_majority),    # to match majority class
                                 random_state=123) #
    
    df_upsampled = pd.concat([df_majority, df_minority_upsampled])

    df_upsampled = spark.createDataFrame(df_upsampled)  
    return df_upsampled

In [0]:
def undersampleDataset(df):
    from sklearn.utils import resample
    df = df.toPandas()
    # Separate majority and minority classes
    df_majority = df[df.label==0]
    df_minority = df[df.label==1]

    # Upsample minority class
    df_majority_downsampled = resample(df_majority, 
                                 replace=False,     # sample with replacement
                                 n_samples=len(df_minority),    # to match majority class
                                 random_state=123) #
    df_downsampled = pd.concat([df_majority_downsampled, df_minority])

    df_downsampled = spark.createDataFrame(df_downsampled)  
    return df_downsampled

#Model Training

In [0]:
#Train the model with the data as is for now
lr = LogisticRegression(
    maxIter=10,
    regParam=0.01,

)

lrModel = lr.fit(train_data)

classificationReport(lrModel,test_data)


In [0]:
#Train the model with the data as is for now
lr = LogisticRegression(
    maxIter=50,
    #regParam=0.01,

)

lrModel = lr.fit(balanceDataset(train_data))#Balanced dataset is balanced here

classificationReport(lrModel,test_data)


In [0]:
#Train the model with the data as is for now
lr = LogisticRegression(
    maxIter=50,
    #regParam=0.01,

)

lrModel = lr.fit(undersampleDataset(train_data))#Balanced dataset is balanced here

classificationReport(lrModel,test_data)


In [0]:
#build random forest model
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=100, maxDepth=30, maxBins=32)

# Train model with Training Data
rfModel = rf.fit(train_data)

classificationReport(rfModel,test_data)


In [0]:
#Oversampled
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=400, maxDepth=25, maxBins=32)

# Train model with Training Data
rfModel = rf.fit(balanceDataset(train_data))

classificationReport(rfModel,test_data)


In [0]:
train_data

In [0]:
train_oversampled = balanceDataset(train_data)

train_oversampled_weighted = train_oversampled.withColumn(
    "weight",
    when(F.col("label") == 1, 5.0).otherwise(1.0)   # milder weights since already balanced
)

# Create RF classifier with weightCol parameter
rf_weighted = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=400, maxDepth=25, maxBins=32, weightCol='weight')

# Train model with weighted data
rfModel = rf_weighted.fit(train_oversampled_weighted)

classificationReport(rfModel,test_data)

In [0]:
# Assuming rfModel is your trained RandomForestClassifier model
# And feature_cols is a list of your actual feature names,
# e.g., ['age', 'income', 'education_level', 'num_children', 'feature_A',..., 'feature_Z']

feature_importances = rfModel.featureImportances

# Combine feature names with their importance scores
features_with_importance = []
for i, importance in enumerate(feature_importances):
    features_with_importance.append((feature_cols[i], importance))

# Sort the features by importance in descending order
features_with_importance.sort(key=lambda x: x[1], reverse=True)

# Get the top 10 features
top_10_features = features_with_importance[:20]
newFeatures = []

print("Top 20 Feature Importances:")
for name, importance in top_10_features:
    newFeatures.append(name)
    print(f"- {name}: {importance:.4f}")

In [0]:
train_data_lessfeatures, test_data_lessfeatures = specifySubsetOfFeaturesToUse(data_Normalized, newFeatures)

rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=1000, maxDepth=25, maxBins=32)

# Train model with Training Data
rfModel = rf.fit(balanceDataset(train_data_lessfeatures))

classificationReport(rfModel,test_data_lessfeatures)

In [0]:
#Neural Network
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# --------------------------------------------------------------------
# Number of input features!
num_features = train_data.select("features").first()[0].size
# --------------------------------------------------------------------

# Define the network architecture → [input, hidden1, hidden2, ..., output]
# Last number MUST = number of classes
layers = [
    num_features,      # input  layer
    1020,              # hidden layer 1
    512,               # hidden layer 2              
    128,               # hidden layer 3 
    64,                # hidden layer 4              
    32,                # hidden layer 5             
    2                  # output → Number of classes
]

# Create the neural network classifier
mlp = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol="label",
    layers=layers,
    maxIter=2000,          # more iterations usually needed vs RF
    blockSize=128,
    stepSize=0.001,        # learning rate
    solver='l-bfgs',      # 'l-bfgs' or 'gd'
    tol=1e-6,
    seed=777              # for reproducibility
)

# Train on the balanced dataset
mlpModel = mlp.fit(balanceDataset(train_data))

# Evaluate (assuming classificationReport is your custom function)
classificationReport(mlpModel, test_data) 


In [0]:
print("Model Information:")
print(f"Model Type: {type(mlpModel).__name__}")
print(f"Number of Layers: {len(mlpModel.getLayers())}")
print(f"Layer Architecture: {mlpModel.getLayers()}")
print(f"Number of Features: {mlpModel.getLayers()[0]}")
print(f"Number of Classes: {mlpModel.getLayers()[-1]}")
print(f"Max Iterations: {mlpModel.getMaxIter()}")
print(f"Step Size (Learning Rate): {mlpModel.getStepSize()}")
print(f"Solver: {mlpModel.getSolver()}")
print(f"Tolerance: {mlpModel.getTol()}")

In [0]:
#Neural Network
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# --------------------------------------------------------------------
# Number of input features!
num_features = train_data_lessfeatures.select("features").first()[0].size
# --------------------------------------------------------------------

# Define the network architecture → [input, hidden1, hidden2, ..., output]
# Last number MUST = number of classes
layers = [
    num_features,      # input  layer
    1020,              # hidden layer 1
    512,               # hidden layer 2              
    128,               # hidden layer 3 
    64,                # hidden layer 4              
    32,                # hidden layer 5             
    2                  # output → Number of classes
]

# Create the neural network classifier
mlp = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol="label",
    layers=layers,
    maxIter=2000,          # more iterations usually needed vs RF
    blockSize=128,
    stepSize=0.001,        # learning rate
    solver='l-bfgs',      # 'l-bfgs' or 'gd'
    tol=1e-6,
    seed=777              # for reproducibility
)

# Train on the balanced dataset
mlpModel = mlp.fit(balanceDataset(train_data_lessfeatures))

# Evaluate (assuming classificationReport is your custom function)
classificationReport(mlpModel, test_data_lessfeatures) 
